# 2.1 脉冲多普勒雷达原理

**学习目标**
- 理解脉冲多普勒雷达的I/Q信号表示
- 掌握脉冲重复频率(PRF)与最大可测速度的关系
- 了解距离模糊和速度模糊的概念
- 观察目标速度对I/Q信号相位的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 2, pp. 121-160

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### I/Q 信号表示

雷达接收的复信号可以表示为：

$$Z = I + iQ = A e^{i\phi}$$

其中 $I = A\cos\phi$ 是同相分量，$Q = A\sin\phi$ 是正交分量。

### 多普勒频移

当目标有径向速度时，接收信号相对于发射信号产生频移：

$$f_D = \frac{2v_r}{\lambda}$$

其中 $v_r$ 是径向速度（远离雷达为正），$\lambda$ 是雷达波长。

In [ ]:
def generate_doppler_signal(velocity=10.0, wavelength=0.10, prf=1000, n_pulses=64, 
                           signal_noise=30.0):
    """
    生成多普勒信号的时间序列
    
    velocity: 目标径向速度 (m/s)
    wavelength: 雷达波长 (m)
    prf: 脉冲重复频率 (Hz)
    n_pulses: 脉冲数
    signal_noise: 信噪比 (dB)
    """
    # 多普勒频移
    f_d = 2 * velocity / wavelength
    
    # Nyquist 频率（最大无模糊速度）
    v_nyquist = wavelength * prf / 4
    
    # 时间序列
    t = np.arange(n_pulses) / prf
    
    # 复信号 (I + iQ)
    phase = 2 * np.pi * f_d * t
    amplitude = 1.0
    signal = amplitude * np.exp(1j * phase)
    
    # 添加噪声
    if signal_noise < 100:
        noise_power = amplitude**2 / (10**(signal_noise/10))
        noise = np.sqrt(noise_power/2) * (np.random.randn(n_pulses) + 1j * np.random.randn(n_pulses))
        signal = signal + noise
    
    # I 和 Q 分量
    I = np.real(signal)
    Q = np.imag(signal)
    
    return {'t': t, 'I': I, 'Q': Q, 'signal': signal, 'f_d': f_d, 'v_nyquist': v_nyquist}

def compute_spectrum(signal, prf):
    """计算信号的频谱"""
    n = len(signal)
    fft_result = np.fft.fftshift(np.fft.fft(signal))
    freqs = np.fft.fftfreq(n, 1/prf)
    freqs = np.fft.fftshift(freqs)
    power = np.abs(fft_result)**2
    return freqs, power, fft_result

## 2. I/Q 信号时序图

In [ ]:
def plot_doppler_time_series(velocity=20.0, wavelength=0.10, prf=1000, 
                              n_pulses=64, signal_noise=30.0):
    """绘制I/Q信号的时间序列和频谱"""
    
    data = generate_doppler_signal(velocity, wavelength, prf, n_pulses, signal_noise)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: I分量时序
    ax1 = axes[0, 0]
    ax1.plot(data['t']*1000, data['I'], 'b-', linewidth=1.5, label='I分量')
    ax1.set_xlabel('时间 (ms)', fontsize=10)
    ax1.set_ylabel('I 幅度', fontsize=10)
    ax1.set_title('I 分量时间序列', fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # 子图2: Q分量时序
    ax2 = axes[0, 1]
    ax2.plot(data['t']*1000, data['Q'], 'r-', linewidth=1.5, label='Q分量')
    ax2.set_xlabel('时间 (ms)', fontsize=10)
    ax2.set_ylabel('Q 幅度', fontsize=10)
    ax2.set_title('Q 分量时间序列', fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # 子图3: I-Q 相平面
    ax3 = axes[1, 0]
    ax3.plot(data['I'], data['Q'], 'g-', linewidth=0.8, alpha=0.7)
    ax3.scatter([data['I'][0]], [data['Q'][0]], color='green', s=100, zorder=5, label='起点')
    ax3.scatter([data['I'][-1]], [data['Q'][-1]], color='red', s=100, zorder=5, label='终点')
    ax3.set_xlabel('I', fontsize=10)
    ax3.set_ylabel('Q', fontsize=10)
    ax3.set_title('I-Q 相平面 (相位旋转轨迹)', fontsize=11)
    ax3.set_aspect('equal')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    # 子图4: 频谱
    ax4 = axes[1, 1]
    freqs, power, _ = compute_spectrum(data['signal'], prf)
    velocities = freqs * wavelength / 2
    ax4.plot(velocities, 10*np.log10(power + 1e-10), 'b-', linewidth=1.5)
    ax4.axvline(x=velocity, color='red', linestyle='--', label=f'真实速度: {velocity:.1f} m/s')
    ax4.axvline(x=data['v_nyquist'], color='orange', linestyle=':', label=f'Nyquist: ±{data["v_nyquist"]:.1f} m/s')
    ax4.axvline(x=-data['v_nyquist'], color='orange', linestyle=':')
    ax4.set_xlabel('速度 (m/s)', fontsize=10)
    ax4.set_ylabel('功率谱密度 (dB)', fontsize=10)
    ax4.set_title('多普勒功率谱', fontsize=11)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(-2*data['v_nyquist'], 2*data['v_nyquist'])
    
    plt.tight_layout()
    plt.show()
    
    # 打印信息
    print(f"\n=== 雷达参数 ===")
    print(f"雷达波长: {wavelength*100:.1f} cm ({'X' if wavelength < 0.05 else 'C' if wavelength < 0.1 else 'S'}波段)")
    print(f"脉冲重复频率: {prf} Hz")
    print(f"Nyquist速度: ±{data['v_nyquist']:.1f} m/s")
    print(f"\n=== 目标参数 ===")
    print(f"目标速度: {velocity:.1f} m/s")
    print(f"多普勒频移: {data['f_d']:.1f} Hz")
    print(f"信噪比: {signal_noise:.0f} dB")
    
    if abs(velocity) > data['v_nyquist']:
        print(f"\n⚠️ 警告: 目标速度超出Nyquist范围，将发生速度模糊!")

In [ ]:
# 交互式演示
interact_doppler = interact(plot_doppler_time_series,
    velocity=widgets.FloatSlider(min=-100, max=100, step=1, value=20.0, 
                               description='径向速度 (m/s)'),
    wavelength=widgets.FloatSlider(min=0.03, max=0.11, step=0.01, 
                                   value=0.10, description='波长 (m)'),
    prf=widgets.FloatSlider(min=250, max=2000, step=250, value=1000, 
                           description='PRF (Hz)'),
    n_pulses=widgets.IntSlider(min=16, max=256, step=16, value=64, 
                               description='脉冲数'),
    signal_noise=widgets.FloatSlider(min=0, max=50, step=5, value=30.0, 
                                     description='信噪比 (dB)')
)

## 3. 速度模糊概念

当目标速度超过Nyquist速度时，会发生速度模糊（folded velocity），即测得的速度与真实速度不同。

In [ ]:
def illustrate_ambiguity():
    """说明速度模糊概念"""
    wavelength = 0.10  # S-band
    prf = 1000
    v_nyquist = wavelength * prf / 4
    
    # 速度范围
    v_true = np.linspace(-3*v_nyquist, 3*v_nyquist, 500)
    
    # 折叠函数
    def fold_velocity(v, v_nyq):
        v_folded = v % (2 * v_nyq)
        v_folded = np.where(v_folded > v_nyq, 2*v_nyq - v_folded, v_folded)
        v_folded = np.where(v_folded < -v_nyq, -2*v_nyq - v_folded, v_folded)
        return v_folded
    
    v_measured = fold_velocity(v_true, v_nyquist)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(v_true, v_measured, 'b-', linewidth=2)
    ax.axhline(y=v_nyquist, color='red', linestyle='--', label=f'+Nyquist = {v_nyquist:.1f} m/s')
    ax.axhline(y=-v_nyquist, color='red', linestyle='--', label=f'-Nyquist = -{v_nyquist:.1f} m/s')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    ax.fill_between(v_true, -v_nyquist, v_nyquist, alpha=0.2, color='green', label='无模糊区')
    ax.set_xlabel('真实速度 (m/s)', fontsize=12)
    ax.set_ylabel('测量速度 (m/s)', fontsize=12)
    ax.set_title(f'速度模糊示意图 (λ={wavelength*100:.0f}cm, PRF={prf}Hz, Nyquist={v_nyquist:.1f}m/s)', fontsize=12)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-75, 75)
    ax.set_ylim(-30, 30)
    
    # 标注折叠区域
    ax.annotate('折叠区1', xy=(35, 10), fontsize=10, color='red')
    ax.annotate('折叠区2', xy=(-60, -10), fontsize=10, color='red')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== 速度模糊示例 ===")
    print(f"假设真实速度 = 35 m/s，测量速度 = {fold_velocity(35, v_nyquist):.1f} m/s")
    print(f"假设真实速度 = -40 m/s，测量速度 = {fold_velocity(-40, v_nyquist):.1f} m/s")

In [ ]:
illustrate_ambiguity()

## 练习题

1. **Nyquist速度计算**：S波段雷达（λ=10cm），PRF=1000Hz，最大无模糊速度是多少？

2. **速度模糊判断**：某目标测量速度为-15m/s，真实速度可能是多少？

3. **PRF选择**：如果要探测最大100m/s的风速，PRF至少需要多少？

4. **I/Q信号**：当目标静止时，I/Q信号的相位如何变化？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 2, Springer.
- Doviak, R.J., and D.S. Zrnic, 1993: *Doppler Radar and Weather Observations*, 2nd ed., Academic Press.